In [9]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [10]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Mrk-544_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Mrk-544_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     365   (2094, 964)   float32   
  1  IMAGE.ERR     1 ImageHDU        46   (2094, 964)   float32   


In [11]:
image_cut = image[55:155, 0:1890]
image_error_cut = image_error[55:155, 0:1890]

In [12]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.0238
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII = 6583 * (1+z)
l_SII = 6717 * (1+z)
l_OIII = 5007 * (1+z)

In [13]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3, params4):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) +
        sersic_1d(x_hr, params4)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [14]:
seeing = 3.174
sigma = seeing / 2.35 
x_axes = np.arange(0, 100, 1)

In [15]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, params4, a, b):
    func = model_convolved(x, params1, params2, params3, params4)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, params4, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    purple_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    sum_area = blue_area + green_area +purple_area + orange_area
    total_area = total_integral(x, params1, params2, params3, params4, a, b)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (purple_area*100)/sum_area, (orange_area*100)/sum_area]     

## Halpha

In [16]:
mask_HA = (wv > l_HA-8)*(wv < l_HA+8)
image_HA = image_cut[:, mask_HA]
image_error_HA = image_error_cut[:, mask_HA]
radial_profile_HA = np.sum(image_HA, axis = 1)
radial_profile_error_HA = np.sqrt(np.sum(image_error_HA**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Fit_code/Halpha_fit.csv", index_col = 0)

In [17]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -sigma, 
                                                                fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -sigma, 
                                                                fit['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -sigma, 
                                                                fit['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -sigma, 
                                                                fit['Component 4'].iloc[3] + sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Halpha_1s.csv')
df.to_csv(df_path, float_format='%.3f')

21.206717870539116
21.60803089898615
4.222077539277411
4.453899157507356
3.882554345718614
4.84395948146155
1.747847412480268
1.7772721080650729
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    95.667476  42.327264   1.661413   1.241271
Green    3.463010  54.104832   1.457132   1.533048
Purple   0.470522   2.222787  92.877292  18.845977
Orange   0.398993   1.345116   4.004163  78.379705


In [18]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -2*sigma, 
                                                                fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -2*sigma, 
                                                                fit['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -2*sigma, 
                                                                fit['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -2*sigma, 
                                                                fit['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Halpha_2s.csv')
df.to_csv(df_path, float_format='%.3f')

38.872061158848155
39.51462884885679
20.189850062717944
20.963640935556693
16.259093137584156
19.473849672519094
4.031854549347862
4.0868510026456635
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    95.095102  48.465542   2.033291   1.280303
Green    3.952995  47.763590   1.776743   1.596968
Purple   0.516015   2.360299  91.367541  19.516280
Orange   0.435887   1.410569   4.822426  77.606450


In [19]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -3*sigma, 
                                                                fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -3*sigma, 
                                                                fit['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -3*sigma, 
                                                                fit['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -3*sigma, 
                                                                fit['Component 4'].iloc[3] + 3*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Halpha_3s.csv')
df.to_csv(df_path, float_format='%.3f')

57.881013875849206
58.647137125643184
28.037400086208297
28.87315240790268
19.793645086150562
23.1423899315711
5.350784508817558
5.407063790480841
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    92.606351  54.209875   2.402143   1.375800
Green    6.225276  41.943145   2.091145   1.704395
Purple   0.643068   2.420364  89.907463  21.142949
Orange   0.525305   1.426616   5.599249  75.776855


## HBeta

In [20]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Fit_code/Hbeta_fit.csv", index_col = 0)

In [21]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -sigma, 
                                                                fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -sigma, 
                                                                fit['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -sigma, 
                                                                fit['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -sigma, 
                                                                fit['Component 4'].iloc[3] + sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Hbeta_1s.csv')
df.to_csv(df_path, float_format='%.3f')

6.655922699004513
6.452112690799256
3.0585457293104144
3.043434379729214
1.9066979957817431
2.2171396996579755
0.9395821137210687
0.9327918901173767
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    87.772721  43.222294   2.038368   2.548011
Green   10.596923  51.462032   1.279874   0.519093
Purple   1.375955   4.494282  93.121550  26.131264
Orange   0.254401   0.821392   3.560208  70.801633


In [22]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -2*sigma, 
                                                                fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -2*sigma, 
                                                                fit['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -2*sigma, 
                                                                fit['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -2*sigma, 
                                                                fit['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s.csv')
df.to_csv(df_path, float_format='%.3f')

12.128106687733794
11.803314816261143
6.121620278578296
6.092388547291503
8.110478560345143
9.141997888928874
1.8195279116514134
1.8089018898990412
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    86.298899  45.938166   2.426540   2.642595
Green   11.904745  48.721444   1.554813   0.545043
Purple   1.516201   4.516598  91.779090  27.266957
Orange   0.280155   0.823791   4.239557  69.545405


In [23]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -3*sigma, 
                                                                fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -3*sigma, 
                                                                fit['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -3*sigma, 
                                                                fit['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -3*sigma, 
                                                                fit['Component 4'].iloc[3] + 3*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s.csv')
df.to_csv(df_path, float_format='%.3f')


17.585713111103637
17.20159545644958
13.027131146004612
12.89454787737304
10.002644458148842
11.073813817358689
2.936330726654161
2.9247873837182117
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    84.143691  55.647498   2.793031   2.778841
Green   13.746208  39.222810   1.828916   0.564636
Purple   1.781822   4.344064  90.496413  28.665819
Orange   0.328278   0.785628   4.881640  67.990704


## NII

In [24]:
mask_NII = (wv > l_NII-8)*(wv < l_NII+8)
image_NII = image_cut[:, mask_NII]
image_error_NII = image_error_cut[:, mask_NII]
radial_profile_NII = np.sum(image_NII, axis = 1)
radial_profile_error_NII = np.sqrt(np.sum(image_error_NII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Fit_code/NII_fit.csv", index_col = 0)

In [25]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -sigma, 
                                                                fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -sigma, 
                                                                fit['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -sigma, 
                                                                fit['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -sigma, 
                                                                fit['Component 4'].iloc[3] + sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_NII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

5.130070519636471
5.30259398702816
1.4823697604250783
1.5128024903611643
5.314435651300981
6.258084041379977
0.586259564251219
0.5936930286363846
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    88.596020  34.921669   1.586679   2.152713
Green    7.772401  55.063803   1.187788   1.087716
Purple   2.847600   8.047007  93.505240  30.740695
Orange   0.783980   1.967521   3.720293  66.018876


In [26]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -2*sigma, 
                                                                fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -2*sigma, 
                                                                fit['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -2*sigma, 
                                                                fit['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -2*sigma, 
                                                                fit['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_NII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

11.574539862142936
11.891529566942387
7.117280164802876
7.21913820412071
11.384742469282983
13.057129926730534
2.8049397104130454
2.82956448365455
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    85.647095  38.864252   1.979006   2.267932
Green   10.173297  50.611913   1.520823   1.152415
Purple   3.283711   8.466681  92.302505  32.664213
Orange   0.895896   2.057154   4.197666  63.915440


In [27]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -3*sigma, 
                                                                fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -3*sigma, 
                                                                fit['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -3*sigma, 
                                                                fit['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -3*sigma, 
                                                                fit['Component 4'].iloc[3] + 3*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_NII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

14.77149543124625
15.09349794850898
9.789843780098495
9.90533498249635
13.998824697046874
15.739591923062992
3.798243360156581
3.8239110757193746
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    83.435557  42.768082   2.285481   2.365543
Green   11.950692  46.413204   1.773777   1.209419
Purple   3.627320   8.715775  91.114188  34.406673
Orange   0.986432   2.102939   4.826554  62.018365


## SII

In [28]:
mask_SII = (wv > l_SII-8)*(wv < l_SII+8)
x_axes = np.arange(0, 100, 1)
image_SII = image_cut[:, mask_SII] 
image_error_SII = image_error_cut[:, mask_SII]
radial_profile_SII = np.sum(image_SII, axis = 1)
radial_profile_error_SII = np.sqrt(np.sum(image_error_SII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Fit_code/SII_fit.csv", index_col = 0)

In [29]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -sigma, 
                                                                fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -sigma, 
                                                                fit['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -sigma, 
                                                                fit['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -sigma, 
                                                                fit['Component 4'].iloc[3] + sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_SII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

1.317279388294412
1.3521391274923116
0.860760952356287
0.8696676089303057
3.714094292495415
4.060546538822447
0.862750204046314
0.8596128247381571
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    81.184591  36.842781   1.502612   1.921091
Green   14.031453  51.812371   2.896615   4.656505
Purple   3.201421   8.120209  91.320392  27.450356
Orange   1.582535   3.224640   4.280382  65.972048


In [30]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -2*sigma, 
                                                                fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -2*sigma, 
                                                                fit['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -2*sigma, 
                                                                fit['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -2*sigma, 
                                                                fit['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_SII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

6.017691561740035
6.1298742684355485
4.201666041493932
4.23172962561018
6.723969480457342
7.276504924508417
1.6755446881149652
1.6707519660251315
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    78.928033  39.715557   1.671680   1.988086
Green   15.805440  48.555495   3.216782   4.811075
Purple   3.530148   8.415972  90.361878  28.578864
Orange   1.736379   3.312975   4.749660  64.621974


In [31]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -3*sigma, 
                                                                fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -3*sigma, 
                                                                fit['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -3*sigma, 
                                                                fit['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -3*sigma, 
                                                                fit['Component 4'].iloc[3] + 3*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_SII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

7.848137659538294
7.962251543441321
5.8259445061802975
5.860694291883754
9.765878641210927
10.40019713584062
2.720795679743661
2.7159496770675378
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    76.783666  42.508192   2.171635   2.072120
Green   17.526785  45.527550   4.120060   5.039926
Purple   3.820557   8.608215  88.132972  29.853049
Orange   1.868992   3.356043   5.575333  63.034905


## OIII

In [32]:
mask_OIII = (wv > l_OIII-8)*(wv < l_OIII+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Fit_code/OIII_fit.csv", index_col = 0)

In [33]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -sigma, 
                                                                fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -sigma, 
                                                                fit['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -sigma, 
                                                                fit['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -sigma, 
                                                                fit['Component 4'].iloc[3] + sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_OIII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

9.99846522549739
8.480604172623949
3.501672892442124
3.4668767333841424
2.0690154129672154
2.4543008172638823
1.0471046012014875
1.0525318221615054
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    92.717031  71.534710   4.263348   6.513062
Green    5.759648  22.670982   0.047460   0.000007
Purple   1.084081   4.195446  90.482796  24.434489
Orange   0.439240   1.598862   5.206397  69.052441


In [34]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -2*sigma, 
                                                                fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -2*sigma, 
                                                                fit['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -2*sigma, 
                                                                fit['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -2*sigma, 
                                                                fit['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_OIII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

17.72751000925801
15.320525643754399
7.514801333162197
7.32601133368023
8.877749274380776
10.163782289567159
2.576057642504723
2.5854943473740906
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    91.811974  73.862170   5.009195   6.854917
Green    6.464248  20.715616   0.071855   0.000021
Purple   1.227309   3.928929  88.806119  26.642985
Orange   0.496469   1.493285   6.112831  66.502077


In [35]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -3*sigma, 
                                                                fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -3*sigma, 
                                                                fit['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -3*sigma, 
                                                                fit['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -3*sigma, 
                                                                fit['Component 4'].iloc[3] + 3*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_OIII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

26.613255936243277
23.81677737489209
13.55946349826989
12.92774751977753
11.024598850632335
12.36298268934117
3.5099363636191048
3.5198564705886604
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    89.259536  75.417069   5.698693   7.078256
Green    8.414511  19.052438   0.103692   0.000038
Purple   1.659189   4.027621  87.248961  27.902831
Orange   0.666764   1.502872   6.948654  65.018876
